In [39]:
"""
Script pour générer un PDF récapitulatif du budget mensuel.
Utilisation : python generate_pdf.py
"""

import sqlite3
from datetime import datetime
from fpdf import FPDF
import os

DB_PATH = "C:\\Users\\jeand\\Downloads\\suivi budget mensuel\\prisma\\db\\custom.db"
OUTPUT_DIR = "pdf"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [40]:

# ─── Couleurs ───────────────────────────────────────────────
PURPLE = (108, 92, 231)
DARK = (45, 52, 54)
GRAY = (127, 140, 141)
WHITE = (255, 255, 255)
LIGHT_BG = (240, 240, 245)
ORANGE = (253, 203, 110)

CATEGORIES = {
    "courses": "Courses",
    "loyer": "Loyer",
    "loisirs": "Loisirs",
    "transport": "Transport",
    "autre": "Autre",
}

CATEGORY_COLORS = {
    "courses": (108, 92, 231),
    "loyer": (0, 184, 148),
    "loisirs": (225, 112, 85),
    "transport": (253, 203, 110),
    "autre": (116, 185, 255),
}


In [41]:
class BudgetPDF(FPDF):
    def header(self):
        # Fond violet
        self.set_fill_color(*PURPLE)
        self.rect(0, 0, 210, 35, "F")
        # Icône €
        self.set_font("Helvetica", "B", 22)
        self.set_text_color(*WHITE)
        self.set_xy(10, 8)
        self.cell(15, 15, "$", border=1, align="C")
        # Titre
        self.set_xy(28, 6)
        self.set_font("Helvetica", "B", 20)
        self.cell(0, 10, "Suiveur de Budget")
        self.set_xy(28, 17)
        self.set_font("Helvetica", "", 10)
        self.cell(0, 6, "Rapport mensuel des depenses")
        self.ln(30)

    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "", 8)
        self.set_text_color(*GRAY)
        self.cell(0, 10, f"Genere le {datetime.now().strftime('%d/%m/%Y a %H:%M')}", align="C")


def format_euro(amount):
    return f"{amount:,.2f} EUR".replace(",", " ")


def get_data():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    # Toutes les dépenses groupées par mois
    cur.execute("SELECT date, amount, category FROM Expense ORDER BY date DESC")
    all_expenses = cur.fetchall()

    # Tous les budgets
    cur.execute("SELECT month, amount FROM Budget")
    budgets = dict(cur.fetchall())

    conn.close()
    return all_expenses, budgets


def generate_pdf():
    pdf = BudgetPDF()
    pdf.set_auto_page_break(auto=True, margin=20)

    for month_key, data in sorted(months.items(), reverse=True):
        budget_amount = budgets.get(month_key, 0)
        remaining = budget_amount - data["total"]

        pdf.add_page()

        # Section titre du mois
        pdf.set_font("Helvetica", "B", 16)
        pdf.set_text_color(*DARK)
        pdf.set_x(10)
        pdf.cell(0, 10, data["label"])
        pdf.ln(12)

        # Cadre budget
        pdf.set_fill_color(*LIGHT_BG)
        pdf.rect(10, pdf.get_y(), 190, 25, "F")

        y_box = pdf.get_y() + 3
        pdf.set_xy(15, y_box)
        pdf.set_font("Helvetica", "", 10)
        pdf.set_text_color(*GRAY)
        pdf.cell(60, 8, f"Budget : {format_euro(budget_amount) if budget_amount else 'Non defini'}")

        pdf.set_xy(15, y_box + 9)
        total_color = (231, 76, 60) if remaining < 0 else (39, 174, 96)
        pdf.set_text_color(*total_color)
        text = f"Reste : {format_euro(remaining)}" if budget_amount else f"Total : {format_euro(data['total'])}"
        pdf.set_font("Helvetica", "B", 10)
        pdf.cell(60, 8, text)

        # Total dépenses à droite
        pdf.set_xy(130, y_box + 2)
        pdf.set_font("Helvetica", "B", 18)
        pdf.set_text_color(*PURPLE)
        pdf.cell(65, 16, format_euro(data["total"]), align="R")

        pdf.set_y(pdf.get_y() + 20)

        # En-têtes tableau
        pdf.set_font("Helvetica", "B", 9)
        pdf.set_fill_color(220, 220, 225)
        pdf.set_text_color(*DARK)
        pdf.set_x(10)
        pdf.cell(90, 8, "  Categorie", fill=True)
        pdf.cell(45, 8, "Date", fill=True, align="C")
        pdf.cell(55, 8, "Montant", fill=True, align="R")
        pdf.ln()

        # Lignes dépenses
        pdf.set_font("Helvetica", "", 9)
        for date_str, amount, category in data["expenses"]:
            d = datetime.fromtimestamp(date_str / 1000)
            cat_label = CATEGORIES.get(category, category)
            color = CATEGORY_COLORS.get(category, GRAY)

            pdf.set_x(10)
            pdf.set_text_color(*DARK)
            # Pastille couleur
            pdf.set_fill_color(*color)
            pdf.rect(14, pdf.get_y() + 1.5, 3, 5, "F")
            pdf.set_x(20)
            pdf.cell(80, 8, cat_label)

            pdf.set_text_color(*GRAY)
            pdf.cell(45, 8, d.strftime("%d/%m/%Y"), align="C")

            pdf.set_text_color(*DARK)
            pdf.set_font("Helvetica", "B", 9)
            pdf.cell(55, 8, format_euro(amount), align="R")
            pdf.set_font("Helvetica", "", 9)
            pdf.ln()

        # Répartition par catégorie
        pdf.ln(8)
        pdf.set_font("Helvetica", "B", 12)
        pdf.set_text_color(*DARK)
        pdf.cell(0, 8, "Repartition par categorie")
        pdf.ln(10)

        grouped = {}
        for _, amount, category in data["expenses"]:
            grouped[category] = grouped.get(category, 0) + amount

        for cat, total in sorted(grouped.items(), key=lambda x: -x[1]):
            pct = (total / data["total"]) * 100 if data["total"] > 0 else 0
            color = CATEGORY_COLORS.get(cat, GRAY)

            pdf.set_x(15)
            pdf.set_fill_color(*color)
            pdf.rect(15, pdf.get_y() + 1, 4, 5, "F")
            pdf.set_x(22)
            pdf.set_font("Helvetica", "", 9)
            pdf.set_text_color(*DARK)
            pdf.cell(50, 8, CATEGORIES.get(cat, cat))

            # Barre de pourcentage
            pdf.set_fill_color(*color)
            bar_width = pct * 0.7
            pdf.rect(72, pdf.get_y() + 1.5, bar_width, 5, "F")
            pdf.set_fill_color(230, 230, 235)
            pdf.rect(72 + bar_width, pdf.get_y() + 1.5, 70 - bar_width, 5, "F")

            pdf.set_x(145)
            pdf.cell(25, 8, f"{pct:.0f}%", align="R")
            pdf.set_font("Helvetica", "B", 9)
            pdf.cell(40, 8, format_euro(total), align="R")
            pdf.set_font("Helvetica", "", 9)
            pdf.ln()

    # Sauvegarder
    filename = f"budget_rapport_{datetime.now().strftime('%Y%m%d_%H%M')}.pdf"
    print(filename)
    filepath = os.path.join(OUTPUT_DIR, filename)
    pdf.output(filepath)
    print(f"PDF genere : {filepath}")


In [42]:

all_expenses, budgets = get_data()
if not all_expenses:
    print("Aucune depense trouvee dans la base.")
all_expenses, budgets

([(1783036800000, 8.78, 'courses'),
  (1782777600000, 32.33, 'courses'),
  (1782518400000, 8.0, 'loisirs'),
  (1782518400000, 4.0, 'loisirs'),
  (1782432000000, 11.4, 'courses'),
  (1782345600000, 30.99, 'courses'),
  (1781827200000, 12.0, 'loisirs'),
  (1781740800000, 26.18, 'courses'),
  (1781740800000, 16.36, 'courses'),
  (1781654400000, 4.0, 'loisirs'),
  (1781654400000, 12.0, 'loisirs'),
  (1781308800000, 14.0, 'loisirs'),
  (1781308800000, 10.0, 'loisirs'),
  (1781222400000, 12.0, 'loisirs'),
  (1781049600000, 35.36, 'courses'),
  (1780790400000, 10.2, 'transport'),
  (1780790400000, 4.0, 'loisirs'),
  (1780704000000, 30.0, 'autre'),
  (1780704000000, 4.0, 'loisirs'),
  (1780704000000, 9.0, 'loisirs'),
  (1780704000000, 4.0, 'loisirs'),
  (1780358400000, 32.44, 'courses'),
  (1780185600000, 4.0, 'loisirs'),
  (1780185600000, 18.5, 'loisirs'),
  (1780012800000, 27.14, 'courses'),
  (1780012800000, 11.7, 'loisirs'),
  (1779926400000, 9.6, 'loisirs'),
  (1779926400000, 8.0, 'loisir

In [43]:
# Grouper par mois
months = {}
for date_str, amount, category in all_expenses:
    d = datetime.fromtimestamp(date_str / 1000)
    key = d.strftime("%Y-%m")
    label = d.strftime("%B %Y").capitalize()
    if key not in months:
        months[key] = {"label": label, "expenses": [], "total": 0}
    months[key]["expenses"].append((date_str, amount, category))
    months[key]["total"] += amount

months

{'2026-07': {'label': 'July 2026',
  'expenses': [(1783036800000, 8.78, 'courses')],
  'total': 8.78},
 '2026-06': {'label': 'June 2026',
  'expenses': [(1782777600000, 32.33, 'courses'),
   (1782518400000, 8.0, 'loisirs'),
   (1782518400000, 4.0, 'loisirs'),
   (1782432000000, 11.4, 'courses'),
   (1782345600000, 30.99, 'courses'),
   (1781827200000, 12.0, 'loisirs'),
   (1781740800000, 26.18, 'courses'),
   (1781740800000, 16.36, 'courses'),
   (1781654400000, 4.0, 'loisirs'),
   (1781654400000, 12.0, 'loisirs'),
   (1781308800000, 14.0, 'loisirs'),
   (1781308800000, 10.0, 'loisirs'),
   (1781222400000, 12.0, 'loisirs'),
   (1781049600000, 35.36, 'courses'),
   (1780790400000, 10.2, 'transport'),
   (1780790400000, 4.0, 'loisirs'),
   (1780704000000, 30.0, 'autre'),
   (1780704000000, 4.0, 'loisirs'),
   (1780704000000, 9.0, 'loisirs'),
   (1780704000000, 4.0, 'loisirs'),
   (1780358400000, 32.44, 'courses')],
  'total': 322.26},
 '2026-05': {'label': 'May 2026',
  'expenses': [(178

In [44]:
generate_pdf()

budget_rapport_20260706_1037.pdf
PDF genere : pdf\budget_rapport_20260706_1037.pdf


In [45]:
import sqlite3
conn = sqlite3.connect("C:\\Users\\jeand\\Downloads\\suivi budget mensuel\\prisma\\db\\custom.db")
cur = conn.cursor()
cur.execute("SELECT category, amount, date FROM Expense ORDER BY date")
for row in cur.fetchall():
    print(row)
conn.close()

('courses', 45.54, 1770076800000)
('courses', 38.64, 1770595200000)
('loisirs', 12.0, 1770940800000)
('loisirs', 10.0, 1771027200000)
('courses', 14.07, 1771027200000)
('loisirs', 4.0, 1771113600000)
('loisirs', 4.0, 1771113600000)
('courses', 42.54, 1771372800000)
('courses', 19.9, 1771459200000)
('loisirs', 8.0, 1771459200000)
('loisirs', 10.0, 1771804800000)
('courses', 33.25, 1771891200000)
('loisirs', 4.0, 1772064000000)
('loisirs', 4.0, 1772064000000)
('courses', 7.9, 1772150400000)
('loisirs', 12.0, 1772236800000)
('courses', 43.84, 1772323200000)
('courses', 33.45, 1772409600000)
('courses', 20.03, 1772582400000)
('loisirs', 4.0, 1772928000000)
('loisirs', 26.8, 1772928000000)
('loisirs', 4.0, 1773187200000)
('loisirs', 4.0, 1773187200000)
('courses', 52.34, 1773446400000)
('courses', 10.9, 1773446400000)
('courses', 10.1, 1773446400000)
('courses', 18.53, 1773532800000)
('loisirs', 16.0, 1773792000000)
('transport', 4.1, 1773878400000)
('courses', 7.3, 1773964800000)
('courses

In [46]:
import pandas as pd

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM Expense", conn)
conn.close()

df.head()

,id,amount,category,date,createdAt,updatedAt
0,cmqw81ehm0000jwucp8zmmuvg,32.44,courses,1780358400000,1782556370818,1782556370818
1,cmqw83xum0004jwucskni817o,30.00,autre,1780704000000,1782556489246,1782556489246
2,cmqw84dml0005jwuc36y5qavp,4.00,loisirs,1780704000000,1782556509694,1782556509694
3,cmqw84jpd0006jwucissprlzd,9.00,loisirs,1780704000000,1782556517570,1782556517570
4,cmqw84oav0007jwuctpe87ysg,4.00,loisirs,1780704000000,1782556523528,1782556523528


In [47]:
df["date_str"] = pd.to_datetime(df["date"], unit="ms").dt.strftime("%Y/%m/%d %H:%M:%S")
df.tail()

,id,amount,category,date,createdAt,updatedAt,date_str
96,cmqzlqfd40014jw304qgod26c,4.00,loisirs,1772064000000,1782760771912,1782760771912,2026/02/26 00:00:00
97,cmqzlqrgr0015jw3079qh2xgq,7.90,courses,1772150400000,1782760787595,1782760787595,2026/02/27 00:00:00
98,cmqzlr2xd0016jw305hu6eqho,12.00,loisirs,1772236800000,1782760802450,1782760802450,2026/02/28 00:00:00
99,cmr0ohmoq0018jw300c1hr7tv,32.33,courses,1782777600000,1782825866515,1782825866515,2026/06/30 00:00:00
100,cmr4uf6j5001ajw30932t9wip,8.78,courses,1783036800000,1783077694668,1783077694668,2026/07/03 00:00:00


In [48]:
monthly_sums = (
    df.assign(month=pd.to_datetime(df["date"], unit="ms").dt.to_period("M"))
      .groupby("month")["amount"]
      .sum()
      .reset_index(name="total")
)
monthly_sums

,month,total
0,2026-02,269.84
1,2026-03,431.22
2,2026-04,299.63
3,2026-05,442.02
4,2026-06,322.26
5,2026-07,8.78


In [49]:
df.assign(month=pd.to_datetime(df["date"], unit="ms").dt.to_period("M")).groupby("month")["amount"]

In [50]:
monthly = df.assign(month=pd.to_datetime(df["date"], unit="ms").dt.to_period("M")).groupby("month")["amount"].sum()
monthly

month
2026-02    269.84
2026-03    431.22
2026-04    299.63
2026-05    442.02
2026-06    322.26
2026-07      8.78
Freq: M, Name: amount, dtype: float64

In [51]:
pd.Period("2026-06")

Period('2026-06', 'M')

In [52]:
monthly.loc[pd.Period("2026-06")]

np.float64(322.26)

In [53]:
df.assign(month=pd.to_datetime(df["date"], unit="ms").dt.to_period("M")).groupby("month")["amount"].sum().tail()

month
2026-03    431.22
2026-04    299.63
2026-05    442.02
2026-06    322.26
2026-07      8.78
Freq: M, Name: amount, dtype: float64